# Session-3: DFDC + Celeb-DF training

Auto-runs end-to-end on Kaggle T4 x2:
1. Bootstrap (clone repo, install, restore data, CDF symlinks)
2. Build combined DFDC+CDF manifests
3. Subset to 750 clips (250 DFDC + 500 CDF)
4. Extract missing CDF frames
5. Train 15 epochs with `--resume` from session-1's `best_smoke.pt`
6. Save new best checkpoint

Total runtime: ~5h. Result -> `/kaggle/working/best_dfdc_cdf_v3.pt`.

## 1. Bootstrap

In [ ]:
import os, shutil, subprocess

os.chdir("/kaggle/working")
if not os.path.exists("code"):
    subprocess.run(["git", "clone", "https://github.com/Parthivsurya/deepfake-detection.git", "code"], check=True)
os.chdir("/kaggle/working/code")
subprocess.run(["git", "pull", "origin", "main"])

subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"])
subprocess.run(["pip", "install", "-q", "--no-deps", "facenet-pytorch"])

SRC = "/kaggle/input/datasets/parthivsuryakb/dfdc-smoke-artifacts"
assert os.path.exists(SRC), f"attach dfdc-smoke-artifacts first - {SRC} missing"
for name in ["frames", "audio", "manifests"]:
    src_dir = f"{SRC}/{name}_dfdc_smoke"
    if os.path.lexists(name):
        if os.path.islink(name) or os.path.isfile(name):
            os.remove(name)
        else:
            shutil.rmtree(name)
    shutil.copytree(src_dir, name)

# Session-1 checkpoint (we'll --resume from this; session-2's v2 was never pushed)
os.makedirs("checkpoints", exist_ok=True)
shutil.copy(f"{SRC}/best_smoke.pt", "checkpoints/best.pt")

CDF_SRC = "/kaggle/input/datasets/parthivsuryakb/celeb-df-v2"
assert os.path.exists(CDF_SRC), f"attach celeb-df-v2 first - {CDF_SRC} missing"
os.makedirs("/kaggle/working/cdf_root", exist_ok=True)
for sub in ["Celeb-real", "Celeb-synthesis"]:
    nested = f"{CDF_SRC}/{sub}/{sub}"
    link = f"/kaggle/working/cdf_root/{sub}"
    if not os.path.lexists(link):
        os.symlink(nested, link)

print("=== ready ===")
print("pwd:", os.getcwd())
print("frames clips:", len([p for p in os.listdir("frames") if os.path.isdir(f"frames/{p}")]))
print("cdf_root:", os.listdir("/kaggle/working/cdf_root"))
subprocess.run(["git", "log", "--oneline", "-3"])


## 2. Verify GPU

In [ ]:
import torch
print("CUDA :", torch.cuda.is_available())
print("Name :", torch.cuda.get_device_name(0))
print("Count:", torch.cuda.device_count())


## 3. Build combined DFDC + CDF manifests

In [ ]:
%cd /kaggle/working/code
!python scripts/prepare_datasets.py \
    --dataset dfdc celebdf \
    --root /kaggle/input/competitions/deepfake-detection-challenge/train_sample_videos \
           /kaggle/working/cdf_root \
    --out manifests/

import pandas as pd
df = pd.read_csv("manifests/train.csv")
print("train rows:", len(df))
print("by dataset:", df["dataset"].value_counts().to_dict())
print("by label:  ", df["label"].value_counts().to_dict())


## 4. Subset to 250 DFDC + 500 CDF for this session

In [ ]:
%cd /kaggle/working/code
import pandas as pd

for split, n_cdf, n_dfdc in [("train", 500, 250), ("val", 80, 50), ("test", 80, 50)]:
    full = pd.read_csv(f"manifests/{split}.csv")
    cdf = full[full["dataset"] == "celebdf"]
    dfdc = full[full["dataset"] == "dfdc"]

    cdf_real = cdf[cdf["label"] == 0].sample(n=min(n_cdf // 2, (cdf["label"] == 0).sum()), random_state=42)
    cdf_fake = cdf[cdf["label"] == 1].sample(n=min(n_cdf // 2, (cdf["label"] == 1).sum()), random_state=42)
    dfdc_sub = dfdc.sample(n=min(n_dfdc, len(dfdc)), random_state=42) if len(dfdc) else dfdc

    combined = pd.concat([cdf_real, cdf_fake, dfdc_sub], ignore_index=True)
    combined.to_csv(f"manifests/{split}.session3.csv", index=False)
    print(f"{split}: {len(combined)} rows | cdf_real={len(cdf_real)} cdf_fake={len(cdf_fake)} dfdc={len(dfdc_sub)}")


## 5. Identify which clips need extraction

In [ ]:
%cd /kaggle/working/code
import pandas as pd
from pathlib import Path

have = {p.name for p in Path("frames").glob("*") if p.is_dir()}
print(f"already have {len(have)} extracted clips")

for split in ["train", "val", "test"]:
    df = pd.read_csv(f"manifests/{split}.session3.csv")
    missing = df[~df["clip_id"].isin(have)]
    missing.to_csv(f"manifests/{split}.s3.todo.csv", index=False)
    print(f"{split}: {len(missing)}/{len(df)} clips to extract")


## 6. Extract frames (~2 hours)

In [ ]:
%cd /kaggle/working/code
import subprocess
for split in ["train", "val", "test"]:
    print(f"\n=== extracting {split} ===")
    ret = subprocess.run([
        "python", "scripts/extract_frames.py",
        "--manifest", f"manifests/{split}.s3.todo.csv",
        "--out_frames", "frames/", "--out_audio", "audio/",
        "--fps", "4", "--max_frames", "32", "--crop_size", "224",
    ])
    print(f"{split} exit: {ret.returncode}")


## 7. Build combined extracted manifests

In [ ]:
%cd /kaggle/working/code
import pandas as pd
from pathlib import Path

for split in ["train", "val", "test"]:
    parts = []
    p1 = f"manifests/{split}.small.extracted.csv"
    if Path(p1).exists():
        parts.append(pd.read_csv(p1))
    p2 = f"manifests/{split}.s3.todo.extracted.csv"
    if Path(p2).exists():
        parts.append(pd.read_csv(p2))
    full = pd.concat(parts, ignore_index=True).drop_duplicates(subset=["clip_id"])
    plan = pd.read_csv(f"manifests/{split}.session3.csv")
    final = full[full["clip_id"].isin(plan["clip_id"])]
    final.to_csv(f"manifests/{split}.s3.extracted.csv", index=False)
    print(f"{split}: {len(final)} rows ready")


## 8. Train with --resume from session-1 best_smoke.pt

In [ ]:
%cd /kaggle/working/code
import yaml, os

cfg = yaml.safe_load(open("configs/default.yaml"))
cfg["data"]["manifest_train"] = "manifests/train.s3.extracted.csv"
cfg["data"]["manifest_val"]   = "manifests/val.s3.extracted.csv"
cfg["data"]["batch_size"]     = 8
cfg["data"]["num_workers"]    = 2
cfg["train"]["epochs"]        = 15
yaml.safe_dump(cfg, open("configs/session3.yaml", "w"))

RESUME = "/kaggle/input/datasets/parthivsuryakb/dfdc-smoke-artifacts/best_smoke.pt"
print(f"resuming from {RESUME}")

!python scripts/train.py --config configs/session3.yaml --device cuda --resume {RESUME}


## 9. Save best.pt as notebook output

In [ ]:
%cd /kaggle/working/code
import shutil
shutil.copy("checkpoints/best.pt", "/kaggle/working/best_dfdc_cdf_v3.pt")
!ls -lh /kaggle/working/best_dfdc_cdf_v3.pt


## 10. Push as new version of dfdc-smoke-artifacts

In [ ]:
%cd /kaggle/working/code
import json, shutil, os
os.makedirs("/kaggle/working/upload3", exist_ok=True)
shutil.copy("/kaggle/working/best_dfdc_cdf_v3.pt", "/kaggle/working/upload3/best_dfdc_cdf_v3.pt")
with open("/kaggle/working/upload3/dataset-metadata.json", "w") as fp:
    json.dump({"title": "dfdc-smoke-artifacts",
               "id": "parthivsuryakb/dfdc-smoke-artifacts",
               "licenses": [{"name": "CC0-1.0"}]}, fp)
!kaggle datasets version -p /kaggle/working/upload3 -m "v2: DFDC sample + 500 CDF clips, 15 epochs"
